# Getting Started / Overview
The **two ways to run CQL** from a notebook, how they hand results back to Python and SQL, how a library
is written in cells and published to a folder, and how a compiled measure is read without running it.
Everything in the other notebooks uses one of the two doors.

1. **`%%cql` — the cell magic.** A cell whose first line is `%%cql` *is* CQL. No quoting,
   no Python wrapper, editor-width room to write real CQL. A cell **declares**, like a Python `def`:
   running it checks the CQL and runs nothing. A cell with no `library` header joins the notebook's
   **session library**, so a later cell sees its defines, as a SQL cell sees a view. A cell with one *is*
   a library, which a later cell can `include`.
2. **`cql(...)` — the Python API.** A lazy query handle you can inspect, parameterize, reuse and run.
   `cql()` with no argument is the session library, so a Python cell runs what the cells declared.

Write CQL in the magic; run it, and hand results on, from Python. Use `cql("...")` when the CQL is a
value your Python code is manipulating.

**Runs on serverless — nothing to attach, nothing to configure.** Just **Run All**. The one
prerequisite is a workspace where CQL on Databricks is deployed.

## 1. The client
The client, `cql_databricks` — compile requests, result frames, discovery, OAuth and the `%%cql` magic —
comes from this notebook's base environment, **CQL on Databricks**, which the deploy builds with the client
wheel in it. There is nothing to install. For a notebook of your own, pick **CQL on Databricks** under
*Base environment* in the notebook's Environment pane, unless your workspace already makes it the default.

## 2. Load the magic
One cell. `%reload_ext` registers `%%cql` on this notebook's IPython shell, and loads it again when the
cell runs again; the import brings in `cql()`. Everything else — the Compiler
Service endpoint, OAuth, the `cql_*` Unity Catalog function catalog — is discovered on first use.

**Expected output:** nothing. the cell is silent on success; an error here means the notebook is
not on the **CQL on Databricks** base environment.

In [0]:
%reload_ext cql_databricks.compiler
from cql_databricks.compiler import cql

In [0]:
# Start clean, so Run All gives the same output however often it runs: drop any earlier mount and any
# library a `%%cql` cell declared (CLIENT-SESSION-RESET). Section 16 redefines a library in place, so a
# stale draft from an earlier run would otherwise answer the first include.
from cql_databricks.compiler import reset

reset(spark)

## 3. Your first CQL cell
The `%%cql` cell declares `Answer`. The Compiler Service checks it, and nothing runs yet: like a
Python `def`, the define is now available, and closing the notebook here would cost nothing more.

The Python cell after it runs it. `cql()` is the session library the cell joined, and `exec()`
compiles it to Spark SQL and runs it on this Spark session.

**Expected output:** the `%%cql` cell prints `added to the session library`. The Python cell shows
one table, one row.

| `__subject_id` | `__val` |
|---|---|
| `*` | `42` |

In [0]:
%%cql
define "Answer": 6 * 7

In [0]:
from cql_databricks.compiler import VALUE_COLUMN, cql

display(cql().exec("Answer"))

## 4. The result shape — always the same two columns
Every define result carries the subject key `__subject_id` and the payload column `__val`.
**A define names the OBJECT — the view or table — never a column**, so you never have to guess
what your result column is called, and two defines always arrive with identical schemas.

The define above declares no `context`, so it resolves to CQL's `Unfiltered` context: there is
no subject, `__subject_id` is the `'*'` seed, and the single row *is* the answer. **Scenarios / Quality Measurement** shows the same shape with one row per patient.

A **list** is one value like any other, so a list-valued define is still one row per subject — the
whole list arrives as an array in `__val`. (A define whose result is a *stream of resources*, such as
a FHIR retrieve, lowers instead to one row per element plus an `__ordinal` column, because CQL lists
are ordered and the order has to be carried as data. You will see that shape in **Getting Started / CQL Language Tour**.)

## 5. Hand the result to Python
`exec("BirthYears")` returns an ordinary DataFrame: `.filter()`, `.join()`, `.write` — whatever you
would do with any other frame. That hand-off is the whole integration story, and it is what keeps
CQL from being a walled garden.

**Expected output:**
```
DataFrame -> ['__subject_id', '__val']
BirthYears = [1990, 2010, 1975]
```
Note the order is the order written, not sorted — CQL lists are ordered.

In [0]:
%%cql
define "BirthYears": { 1990, 2010, 1975 }

In [0]:
ages = cql().exec("BirthYears")

print(type(ages).__name__, "->", ages.columns)
print("BirthYears =", ages.collect()[0][VALUE_COLUMN])

## 6. …and `%sql` can read it too
A run of the session library also lands every define as a **session temp view named after the
define**, so a plain SQL cell needs nothing from CQL — it just queries the result. `Answer` from section 3 is still there; the
value inside it is, as always, the column `__val`.

**Expected output:** one row — `subject = *`, `answer = 42`.

In [0]:
%sql
SELECT __subject_id AS subject, __val AS answer FROM Answer

## 7. Choose which defines to evaluate
`exec()` takes the defines to run. One name returns one DataFrame; several return a
`{define: DataFrame}` **dict**. With none, it runs the library's eCQM populations if it has any, else
**every** define.

**Expected output:** `dict_keys(['Screened', 'Unscreened'])`, then `Screened = 840` and
`Unscreened = 360`. `Eligible` is defined and used, but not requested, so it is not returned.

In [0]:
%%cql
define "Eligible":   1200
define "Screened":   840
define "Unscreened": "Eligible" - "Screened"

In [0]:
cohort = cql().exec("Screened", "Unscreened")

print(cohort.keys())
for name, frame in cohort.items():
    print(name, "=", frame.collect()[0][VALUE_COLUMN])

## 8. One session library — later cells see earlier defines
Every cell so far had no `library` header, so each one joined the notebook's **session library**
(`SVC-NOTEBOOK-SESSION-SCOPE`). A later cell refers to an earlier define by name, with nothing to
import. Redefining a name **replaces** it, the way re-running a SQL cell's `CREATE OR REPLACE VIEW`
does, and every define that uses it follows.

Give a cell a `library` header when you want a separate, includable library instead (section 13).

**Expected output:** `Total = 1200` (`Screened` + `Unscreened` from section 7). After `Eligible` is
redefined as `1500`, the next run prints `Total = 1500`, and the `%sql` cell reads `total = 1500`:
`Unscreened` is now `660`, so `Total` moved with it.

In [0]:
%%cql
define "Total": "Screened" + "Unscreened"

In [0]:
print("Total =", cql().exec("Total").collect()[0][VALUE_COLUMN])

In [0]:
%%cql
define "Eligible": 1500

In [0]:
# The views are re-created by a run, so run again before SQL reads them.
print("Total =", cql().exec("Total").collect()[0][VALUE_COLUMN])

In [0]:
%sql
SELECT __val AS total FROM Total

## 9. CQL as a Python value
`cql(source)` returns a **lazy query handle**. Nothing runs until you ask it for something, and the
compile is memoized — inspecting a library and then running it costs one round trip, not two.

A string with no `library` header joins the same session library as the cells above. `cql()` checks
at once that the whole library still compiles. `exec()` with no targets runs only the string's own
defines, and `defines()` lists the whole library.

This is the door to use when the CQL is a *value* your code holds: read from a file, resolved from a
registry, or looped over.

**Expected output:**
```
its defines : ['Answer2', 'Doubled']
library     : ['Answer', 'Answer2', 'BirthYears', 'Doubled', 'Eligible', 'Screened', 'Total', 'Unscreened']
populations : []
Doubled     = 84
```
`populations` is empty because this library has no well-known population define, such as
`Initial Population`. Section 15 shows one that does.

In [0]:
q = cql('''
define "Answer2": 6 * 7
define "Doubled": "Answer2" * 2
''', spark=spark)

print("its defines :", q.default_targets())
print("library     :", sorted(q.defines()))
print("populations :", q.populations())
print("Doubled     =", q.exec("Doubled").collect()[0][VALUE_COLUMN])

## 10. Parameters — the same handle, specialized
A CQL `parameter` with a `default` is the mechanism every measure uses for its measurement period.
The handle is **immutable-fluent**: `with_parameter` returns a NEW handle, so one bound library can
be reused and specialized rather than rebound.

An override is CQL *expression text*, so a Python string passes through verbatim as CQL — which is
how a date, interval or quantity literal is bound (`"@2026-01-01"`, `"40 'mg'"`).
`with_measurement_period("2026-01-01", "2026-12-31")` is the shorthand for the ubiquitous one, and it
widens both bounds to instants so they compare at the precision real resource
timestamps carry — a detail **Getting Started / CQL Language Tour** shows the teeth of.

**Expected output:**
```
default threshold (10) : True
bound threshold  (100) : False
```

In [0]:
scored = cql('''
parameter "Threshold" default 10
define "OverThreshold": 42 > "Threshold"
''', spark=spark)

print("default threshold (10) :", scored.exec("OverThreshold").collect()[0][VALUE_COLUMN])
print("bound threshold  (100) :",
      scored.with_parameter("Threshold", 100).exec("OverThreshold").collect()[0][VALUE_COLUMN])

## 11. The data a measure reads
The measures below read FHIR data. Every `[Resource]` retrieve reads the default environment's table of its
type, in its `data` schema, so there is nothing to load. Authoring the logic in cells changes nothing about how
data reaches it — that is the seam holding still while everything above it moves.

**Expected output:** the environment's name and catalog, then one line per type: its row count and the table
it is read from.

In [0]:
from cql_databricks.resolvers import EnvironmentResolver
from cql_databricks.data_bindings import spark_runner

environment = EnvironmentResolver(spark_runner(spark)).default()
print("environment:", environment.name, "in catalog", environment.catalog)
for t in ("Condition", "Encounter", "Patient"):
    view = f"{environment.schema('data')}.{t.lower()}"
    print(f"{t:12} {spark.table(view).count()} rows -> {view}")

## 12. Connect — with nothing mounted as a library source
A client of your own, from `connect()`, with no `library_path`. There is no folder of CQL to point at,
because the libraries are about to be written in the next two cells.

**Expected output:** `libraries declared so far: []` — an empty session, which is the honest state
of a notebook before anything has been authored.

In [0]:
from cql_databricks.compiler import connect

client = connect(spark)
print("libraries declared so far:", [str(r) for r in client.library_provider.declared()])

## 13. A library in a cell — it declares, it does not run
A helper library first: what does it mean for a patient to have visits during the measurement period?
It goes in its own library because "had an encounter" is not specific to one measure — the next measure
you write wants the same definition to mean the same thing.

A `%%cql` cell is a definition, not a question, like a Python `def`. Running it checks the CQL and
declares the library; it reads no data. A Python cell runs a define when you want its result.

**Expected output:** `declared NotebookVisits|1.0.0 — include it, or run it from Python with cql(library="NotebookVisits")`

In [0]:
%%cql
library NotebookVisits version '1.0.0'

using FHIR version '4.0.1'

include FHIRHelpers version '4.0.1' called FHIRHelpers

parameter "Measurement Period" Interval<DateTime>
  default Interval[@2026-01-01T00:00:00.000, @2026-12-31T23:59:59.999]

context Patient

define "Has Visit During Period":
  exists ([Encounter] E
    where E.period during "Measurement Period")

## 14. The measure — a second cell that `include`s the first
`include NotebookVisits` resolves against **this session**. The cell above is the library it
finds, and its text ships with the compile as a dependency, exactly as a file out of a Git Folder
would. Nothing was written to disk in between.

Like the first cell, it declares and runs nothing. It defines `Initial Population` and other checks,
which run when section 15 calls `exec()`.

**Expected output:** `declared NotebookMeasure|1.0.0 — include it, or run it from Python with cql(library="NotebookMeasure")`

In [0]:
%%cql
library NotebookMeasure version '1.0.0'

using FHIR version '4.0.1'

include FHIRHelpers version '4.0.1' called FHIRHelpers
include NotebookVisits version '1.0.0' called Visits

parameter "Measurement Period" Interval<DateTime>
  default Interval[@2026-01-01T00:00:00.000, @2026-12-31T23:59:59.999]

context Patient

define "Has Visit":
  Visits."Has Visit During Period"

define "Has Any Condition":
  exists ([Condition])

define "Initial Population":
  "Has Visit" and "Has Any Condition"

## 15. The draft is addressable by name
Both cells are libraries the session can list and bind, so the Python API reaches them the same way
it reaches published content. Python is where a cell's defines run, and where the measurement
period is bound to a different year.

Watch the population turn over: in 2025 no patient has both a visit and a condition. The patients seen
in 2025 have no condition on record. The measure that reports 1,223 of 4,000 for 2026 reports 0 of 4,000
for 2025 — the same content, honestly producing both numbers.

**One thing to know before you draft a second measure in the same notebook.** Every define is
materialized as a session temp view named after it, so two measures that both define `Initial Population`
overwrite each other's view and the older frame quietly starts returning the newer measure's rows.
Set `spark.conf.set("spark.cqldatabricks.objectPrefix", "auto")` before connecting to keep each
library's objects apart — worth doing the moment a session holds more than one measure, which is
exactly what authoring in a notebook invites.

**Expected output:**
```
declared: ['NotebookVisits|1.0.0', 'NotebookMeasure|1.0.0', 'FHIRHelpers|4.0.1', 'HypertensionBPControl|1.0.0', 'ColorectalScreening|1.0.0', 'MeasureCommon|1.0.0', 'DiabetesHighOrMissingA1c|1.0.0', 'AdultFluImmunization|1.0.0']
2026: initial_population=1223
2025: initial_population=0
```

In [0]:
from cql_databricks.compiler import VALUE_COLUMN

print("declared:", [str(r) for r in client.libraries()])

measure = client.cql(library="NotebookMeasure")

for year in (2026, 2025):
    pops = measure.with_measurement_period(f"{year}-01-01", f"{year}-12-31").exec()
    initial_pop = pops["Initial Population"].collect()
    count = sum(1 for r in initial_pop if r[VALUE_COLUMN])
    print(f"{year}: initial_population={count}")

## 16. Editing a definition
Re-running a cell **replaces** the library it declared. Make a simple edit to the visits library and
the measure that includes it answers differently on the next run — no re-import, no restart, no publish step.

That is also the one hazard worth naming out loud: **the session is the source of truth here, not the
page.** A cell you edited but never re-ran is not the definition the next compile sees. If a number
ever surprises you mid-draft, re-run the library cell before believing it.

The edit below widens the visits check to count a visit up to a year after the period ends. Run for 2025,
the measure counted no patients in section 15. With the edit it also counts the 2026 visits, and every
patient with a condition had one.

**Expected output:** `widened_2025: initial_population=1223`

In [0]:
# The same library, redeclared from Python rather than a cell: `register_cell` is what `%%cql` calls, so a
# generated or parameterized variant of a definition is a first-class way to declare one.
from pyspark.sql import functions as F

from cql_databricks.compiler import register_cell

draft = client.library_provider.source("NotebookVisits").cql
# Edit: count a visit up to a year after the period ends. Fail loud if the text to edit is not there.
ORIGINAL = 'E.period during "Measurement Period"'
WIDENED = 'E.period during Interval[start of "Measurement Period", end of "Measurement Period" + 1 year]'
if ORIGINAL not in draft:
    raise ValueError(f"NotebookVisits no longer contains {ORIGINAL!r}; update this edit")
register_cell(draft.replace(ORIGINAL, WIDENED))

edited_measure = client.cql(library="NotebookMeasure").with_measurement_period("2025-01-01", "2025-12-31")
initial_pop = edited_measure.exec("Initial Population").collect()
count = sum(1 for r in initial_pop if r[VALUE_COLUMN])
print("widened_2025: initial_population=", count, sep="")

## 17. Libraries in a folder
### Publishing the draft
A published library is the same text in a file. Writing the two cells out and mounting that directory
gives a folder like the examples' `measures/` folder below — and the measure resolves by name from it with
no change to a single line of CQL.

In a real workflow the directory is your measures repo as a **Databricks Git Folder**, and publishing
is a commit. The point here is only that nothing about the draft was notebook-shaped: the cell was
always the file.

**You publish the draft you have.** The previous section left `NotebookVisits` sitting with the
widened-window edit, and writing the session out now would publish exactly that — so the experiment
is put back first. Worth doing deliberately rather than discovering it in a file: the notebook holds
one current definition per name, not a history you can pick a version out of.

Cells still resolve ahead of the folder, so `clear()` is what hands the names back to the published
copies once both exist.

The reverse never happens. A measure resolved **from the folder** has its whole `include` closure
resolved there too, so a same-named cell cannot change what a published measure means — overriding is
something you opt into by writing the measure as a cell, never something that happens to a measure you
merely ran. A notebook that could quietly redefine published content would make every number out of it
impossible to cite.

**Expected output:**
```
published: ['NotebookMeasure-1.0.0.cql', 'NotebookVisits-1.0.0.cql']
from the folder: initial_population=1223
```

In [0]:
import pathlib
import tempfile

# Put the original NotebookVisits definition back; the widened window was an experiment.
register_cell(client.library_provider.source("NotebookVisits").cql.replace(WIDENED, ORIGINAL))

published = pathlib.Path(tempfile.mkdtemp(prefix="cql_databricks-draft-"))
for ref in client.library_provider.declared():
    (published / f"{ref.name}-{ref.version}.cql").write_text(
        client.library_provider.source(ref.name).cql)
print("published:", sorted(p.name for p in published.glob("*.cql")))

# Mount the folder, then drop the drafts so the published copies are what the names resolve to.
from_folder = connect(spark, library_path=str(published))
from_folder.library_provider.clear()

pops = from_folder.cql(library="NotebookMeasure").exec("Initial Population")
print(f"from the folder: initial_population={pops.filter(F.col(VALUE_COLUMN)).count()}")

### The examples' folder — the Git Folder setup
A real measure is not text you paste: it is a library that `include`s one or more others, published and
versioned in a repo.

The compile service resolves **none** of those includes — deliberately. It is a dumb endpoint: the
caller resolves the closure and ships every dependency, and the service only merges what it is
handed. It reads no filesystem, no registry and no network. So the resolver
lives on the client, next to the files.

`library_path` mounts one or more directories of `.cql` as the session's library source, and a
measure in them runs **by name**. In a real deployment that
directory is a **Databricks Git Folder** — your measures repo, synced by the platform — or a Unity
Catalog Volume. It is an ordinary directory either way, so there is no platform code involved:

```python
client = connect(spark, library_path="/Workspace/Users/me/my-measures/cql")
```

Roots are **ordered and earlier wins**, so a work-in-progress directory shadows a vendored content
repo: `library_path=["/Workspace/Users/me/wip", "/Volumes/acme/content/measures"]`.

Here we point it at the `measures/` folder published beside the example folders — the example measures, which the
deploy also published into the environment's knowledge. The path is derived from this notebook's own location, so the cell works unchanged
wherever the folder was copied to.

**Expected output:**
```
library_path: /Workspace/…/examples/measures
  AdultFluImmunization|1.0.0
  ColorectalScreening|1.0.0
  DiabetesHighOrMissingA1c|1.0.0
  FHIRHelpers|4.0.1
  HypertensionBPControl|1.0.0
  MeasureCommon|1.0.0
populations of HypertensionBPControl: ['Initial Population', 'Denominator', 'Denominator Exclusions', 'Numerator']
```
That last line is the whole feature. Binding by name resolved the measure, walked its `include`
closure to find the helper libraries, and shipped all of them to the compiler — and **no data was
touched**, because resolving and compiling a measure is not the same act as running it.

In [0]:
import pathlib

from cql_databricks.compiler import connect

_nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
MEASURES = (pathlib.Path("/Workspace") / pathlib.PurePosixPath(_nb).parent.parent.relative_to("/")
            / "measures")
if not MEASURES.is_dir():
    raise FileNotFoundError(f"no measures folder at {MEASURES} — it is published beside the example folders")

client = connect(spark, library_path=str(MEASURES))
print("library_path:", MEASURES)
for ref in client.libraries("*"):
    print(" ", ref)

measure = client.cql(library="HypertensionBPControl").with_measurement_period("2026-01-01", "2026-12-31")
print("populations of HypertensionBPControl:", measure.populations())

## 18. What did *not* happen
No endpoint. No token. No cluster. No `%scala` cell. No catalog or schema named anywhere in this
notebook. A single deployment matching the naming convention resolves itself:

| What | How it was resolved |
|---|---|
| Compiler Service endpoint | looked up in the workspace App registry by name |
| Authentication | your ambient identity, exchanged for an App token (RFC 8693) |
| The `cql_*` operator catalog | read out of `system.information_schema.routines` |
| FHIR data / value sets | the default environment: the catalog tagged `cql_databricks_environment_default`, whose `data` and `knowledge.valueset` tables are read by name — only when the CQL retrieves data |

Each is overridable (`connect(app_name=…, environment=…, data=…, terminology=…)`) — but you have to *want*
to.

The library source is the one thing you *do* name, and that is deliberate: which measures you mean
is a decision, not a convention.

## 19. And it's inspectable — the define graph, ELM and the Spark plan
Everything above ran. A measure can also be read **without running it**: CQL compiles once into
artifacts you can inspect before — and without — executing anything: the neutral **define DAG**, the **ELM**
interchange IR, and the underlying **Spark physical plan**. This is the surface an external pipeline or
governance layer builds on, and the reason a measure can be reviewed before it is trusted with data.

All three are read off **one measure, the example measure Scenarios / Quality Measurement runs** — and no row
of clinical data is read in this section. The physical plan needs the data *contract* rather than the data, so
each retrieve reads a **typed stub** and the plan is printed, never run.

### The measure under inspection
The example measure, `HypertensionBPControl`, read from the `measures/` folder published
beside this notebook. It includes `MeasureCommon`, which resolves from the same folder.
Nothing runs: the handle below is **lazy**, and when it compiles the result is memoized, so every section
shares one round trip. The measure has a default measurement period parameter, so the handle passes one.

**Expected output:** `HypertensionBPControl` and `MeasureCommon`, each with its CQL size.

In [0]:
import pathlib

from cql_databricks.compiler import connect, translate

_nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
MEASURES = (pathlib.Path("/Workspace") / pathlib.PurePosixPath(_nb).parent.parent.relative_to("/")
            / "measures")
if not MEASURES.is_dir():
    raise FileNotFoundError(f"no measures folder at {MEASURES} — it is published beside the example folders")

client = connect(spark, library_path=str(MEASURES))
SOURCES = {name: (MEASURES / f"{name}.cql").read_text() for name in ("HypertensionBPControl",
                                                                      "MeasureCommon")}
for name, cql in SOURCES.items():
    print(f"{name} — {len(cql)} chars of CQL")
q = client.cql(library="HypertensionBPControl").with_measurement_period("2026-01-01", "2026-12-31")

### The neutral define DAG (`dag()` / `describe()`)
What an orchestrator reads to build its own dataset graph *before* committing to any execution
engine: the defines, the dependency edges between them, the cost-chosen materialization boundaries,
the auto-detected population set, and the compiler version pin that makes a result reproducible.

It is **neutral** in the sense that matters: nothing in it names Spark.

**Expected output:**
```
defines             : [... approximately 14 defines including population definitions,
                       hypertension checks and blood pressure criteria ...]
dependencies        : {'Initial Population': [...],
                       'Numerator': ['Most Recent Diastolic', 'Most Recent Systolic'],
                       'Denominator': ['Initial Population'], ...}
populations         : ['Initial Population', 'Denominator', 'Denominator Exclusions', 'Numerator']
materialized_defines: [...]   # the barriers the cost model chose
boundaries          : [...]
compiler_version    : '...'   # the pin the run is reproducible against
```
The graph shows the hypertension measure's population defines at the top level, with blood pressure reads,
condition checks, and encounter eligibility flowing in. The included libraries' defines are named with their
library prefix: the includes are part of the graph.

In [0]:
graph = q.dag()
print("defines             :", graph.defines)
print("dependencies        :", graph.dependencies)
print("populations         :", q.populations())
print("materialized_defines:", graph.materialized_defines)
print("boundaries          :", graph.boundaries)
print("compiler_version    :", graph.compiler_version)

### ELM — the interchange IR
`translate()` returns the ELM JSON the engine's front end produces from CQL — HL7's standard
interchange form for CQL logic.

The library translated here is `MeasureCommon`, one the measure includes. Its only include is
FHIRHelpers, which is content like any library, so its text is passed along with it.

**Expected output:** a byte count of several thousand (the library has multiple functions and queries),
the statement names, and the first 600 characters of one define's expression tree — showing the `Query` and
filter logic over FHIR retrieves, which is the CQL you read, spelled out in ELM.
```
ELM bytes  : ...
statements : ['Patient', 'Age At End Of Period', 'Qualifying Encounters', ...]
```
(`Patient` is the implicit context statement the translator adds.)

In [0]:
import json
elm = translate(SOURCES["MeasureCommon"], includes=[(MEASURES / "FHIRHelpers.cql").read_text()])
elm_obj = json.loads(elm)
print("ELM bytes  :", len(elm))
print("statements :", [s["name"] for s in elm_obj["library"]["statements"]["def"]])
print("\nELM for 'Qualifying Encounters':")
encounters = next(s for s in elm_obj["library"]["statements"]["def"] if s["name"] == "Qualifying Encounters")
print(json.dumps(encounters["expression"], indent=2)[:600], "...")

### The Spark physical plan (`explain`) — of this measure
Under all of it, the compiled SQL is ordinary Spark, and the measure's OWN plan is readable without
data: a plan needs the data **contract**, not the rows. So each retrieve reads a **typed stub**: a
DataFrame on the data contract every data table returns, a `resource` struct of the type the compiler's FHIR
model defines (`client.resource_schema(type)`), with a single all-null row instead of a clinical record. A
`DataFrameDataResolver` hands them to a second client; value sets still come from the environment's
`knowledge.valueset` table.

One row rather than none, because an empty relation is something the optimizer is entitled to fold
away — the plan would collapse to nothing and prove nothing. The row is a **plan placeholder** and is
never read: `explain` plans, it does not execute.

**Expected output:** the stubbed types — the hypertension measure reads Patient, Encounter, Condition (for
hypertension and other diagnoses), Observation (for blood pressure readings), and Procedure (for
dialysis/transplant).
```
stubs: ['Condition', 'Coverage', 'Encounter', 'Observation', 'Patient', 'Procedure']
```

In [0]:
from pyspark.sql.types import StructField, StructType

from cql_databricks.resolvers import DataFrameDataResolver


def stub(fhir_type):
    """The data contract, a `resource` struct, with a plan placeholder behind it: one all-null row
    carrying the compiler's struct for the type. Read from the model, never inferred: there are no rows to
    infer a schema from, which is the sharpest version of why the contract is written down."""
    contract = StructType([StructField("resource", client.resource_schema(fhir_type).struct)])
    return spark.createDataFrame([(None,)], contract)


STUBS = {t: stub(t) for t in ("Condition", "Coverage", "Encounter", "Observation", "Patient", "Procedure")}
planned = connect(spark, library_path=str(MEASURES), data=DataFrameDataResolver(STUBS, name="stubs"))
print("stubs:", sorted(STUBS))

### Read the plan
`exec()` runs the compiled script's statements — every one a `CREATE OR REPLACE TEMP VIEW`, including
the cost-chosen barriers — and hands back a **lazy** frame per population. Views are lazy, so this
defines the computation without performing it, and `.explain()` prints what Spark would do if asked.

Every define's frame is independently `.explain()`-able, which is also how you read a materialized
barrier's full plan.

**Expected output:** a `== Physical Plan ==` block for `Initial Population` — the `Encounter`, `Condition`,
and `Observation` retrieves with their filters through the `cql_*` UC functions, and the joins that flow
age, hypertension status, and encounter eligibility into the population. No private runtime anywhere in it,
and nothing your existing Spark tooling cannot read.

In [0]:
planned_q = planned.cql(library="HypertensionBPControl").with_measurement_period("2026-01-01", "2026-12-31")
pops = planned_q.exec()   # no targets -> the populations, one lazy frame each
pops["Initial Population"].explain(mode="formatted")

Three artifacts from one compile of one measure, none of them needing a row of data. Between them they
are what lets a measure be reviewed, scheduled, shared and audited as an asset rather than run as a
script.

---

### Where next
- **Getting Started / CQL Language Tour** — what the language can express: null handling, precision dates,
  UCUM units, timing relations, equality vs equivalence, terminology filtering, polymorphic choice types.
- **Getting Started / Data and Knowledge Resolution** — every place data, value sets, libraries and results can come from.
- **Scenarios / Quality Measurement** — a published measure run over FHIR data, producing a rate.

Back to **Start Here** for the index.

In [0]:
# Every cell above ran, so the notebook is green. There are no value assertions here by design: the
# narrative states each expected result, and a compile or runtime failure fails its own cell loudly
# rather than being collected into a summary at the end.
import json
dbutils.notebook.exit(json.dumps({"ok": True, "notebook": "getting_started/01_overview"}))